
ref.
 https://stackoverflow.com/questions/53200747/how-can-one-switch-off-the-checkpoint-notification


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pymatgen.core import Element
from copy import deepcopy
import seaborn as sns
import os
from progressbar import ProgressBar

pd.set_option("display.max_rows", 1000)

In [ ]:
def get_data():
    """データ取得

    Returns:
        pd.DataFrame: データ。
        [str]: 元素名リスト。
        [str]: 目的変数名リスト。
        [str]: メタカラム名リスト。
        int: 目的へs縫うの分割数。
    """
    import json
    filepath = os.path.join("../data_calculated/hea4_phys_condition.json")
    with open(filepath, "r") as f:
        cond = json.load(f)
    ndiv = cond["NDIV"] # digitizeする分割数。

    # 加工済みデータの読み込み
    element_labels = []
    for i in range(4):
        element_labels.append("element{}".format(i+1))
    target_names = ['M', 'TC', 'R', ]
    meta_names = ['heakey', ]
    filepath = os.path.join("../data_calculated/hea4_phys.csv")
    dfraw = pd.read_csv(filepath)
    return dfraw, element_labels, target_names, meta_names, ndiv


g_dfraw, g_element_labels, g_target_names, g_meta_names, g_ndiv = get_data()


In [ ]:
g_dfraw


#### データ加工
transactionへの変換

In [ ]:
def convert_to_transaction(df, element_labels,
                           feature_id_labels=["M_id", "TC_id", "R_id",
                                              "group_mean_id", "group_std_id",
                                              "row_mean_id", "row_std_id",
                                              'n_group3', 'n_group4', 'n_group5', 'n_group6', 'n_group7',
                                              'n_group8', 'n_group9', 'n_group10', 'n_group11', 'n_group12',
                                              'n_group13', 'n_group14', 'n_group15',
                                              ]):
    """transactionへの変換。

    Args:
        df (pd.DataFrame): data.
        element_labels ([str])): 元素名リスト
        feature_id_labels ([str]), optional): itemとして使用するカラム名リスト. Defaults to ["M_id", "TC_id", "R_id", "group_mean_id", "group_std_id", "row_mean_id", "row_std_id", 'n_group3', 'n_group4', 'n_group5', 'n_group6', 'n_group7', 'n_group8', 'n_group9', 'n_group10', 'n_group11', 'n_group12', 'n_group13', 'n_group14', 'n_group15', ].

    Returns:
        list: transaction
    """

    discretevalues = {}

    for idname in element_labels:
        discretevalues[idname] = df[idname].values.tolist()

    for idname in feature_id_labels:
        value_list = []
        for value in df[idname].values.tolist():
            if idname == "R_id":
                valueid = "{}=={}".format(idname, value)
            elif idname in ["M_id", "TC_id"]:
                if value > 1:
                    valueid = "{}=={}".format(idname, value)
                else:
                    valueid = ""  # ignore M and TC
            elif idname.startswith("group") or idname.startswith("row"):
                if value > 0:
                    valueid = "{}=={}".format(idname, value)
                else:
                    valueid = ""
            elif idname.startswith("n_"):
                if value > 1:
                    valueid = "{}=={}".format(idname, value)
                else:
                    valueid = ""
            else:
                valueid = "{}=={}".format(idname, value)
            value_list.append(valueid)
        discretevalues[idname] = value_list

    df_discrete = pd.DataFrame(discretevalues)

    transaction = []
    for values_raw in df_discrete.values:
        values_raw = values_raw.tolist()
        values = list(filter(None, values_raw))  # listから””を除く。
        transaction.append(values)

    return transaction


### 特徴の特定

M vs TCを書く。各元素が分布していることが分かる。

In [ ]:
def show_prop_in_2d(dfraw, elementslist, x="M", y="TC", ms=2, alpha=0.5):
    """x,y面でdffrawの表示を行う。 

    Args:
        dfraw (pd.DataFrame): データ。
        elementslist ([[str]]): 元素名リスト。
        x (str, optional): x軸名. Defaults to "M".
        y (str, optional): y軸名. Defaults to "TC".
        ms (int, optional): marker size. Defaults to 2.
        alpha (float, optional): marker alpha value. Defaults to 0.5.
    """

    fig, ax = plt.subplots(figsize=(5, 5))
    dfraw.plot(x="M", y="TC", ax=ax, style=".",
               ms=0.1, c="gray", alpha=0.5, label=None)
    print("all", dfraw.shape)
    colors = ["red", "blue", "green"]
    for i, elms in enumerate(elementslist):
        dfq = dfraw.copy()
        for elm in elms:
            dfq = dfq[dfq["elements"].str.contains(",{},".format(elm))]
        print(elms, dfq.shape, colors[i])
        dfq.plot(x="M", y="TC", ax=ax, style=".", ms=ms,
                 alpha=alpha, c=colors[i], label=str(elms))
    plt.ylabel("TC")
    plt.legend()
    plt.tight_layout()


In [ ]:
show_prop_in_2d(g_dfraw, [["Mn"], ["Fe"], ["Co"]], ms=2, alpha=0.3)


In [ ]:
show_prop_in_2d(g_dfraw, [["Mn", "Fe"], ["Mn", "Co"],
                ["Fe", "Co"]],  ms=5, alpha=0.3)


(Mid,TCid)で特定した領域はどのような特徴を持つのか？

In [ ]:
def show_df_cell(dfraw, dfq, x="M", y="TC"):
    """dfrawとdfqの表示。

    Args:
        dfraw (pd.DataFrame): データ１。
        dfq (pd.DataFrame): データ２。
        x (str, optional): x軸名. Defaults to "M".
        y (str, optional): y軸名. Defaults to "TC".
    """
    fig, ax = plt.subplots(figsize=(5, 5))
    dfraw.plot(x=x, y=y, ax=ax, style=".",
               ms=1, c="gray", alpha=1, label=None)
    dfq.plot(x=x, y=y, ax=ax, style=".", ms=5,
             alpha=1, c="red", label="Select")


def choose_df(dfraw, Mid=7, TCid=5):
    """cellの選択。

    Args:
        dfraw (pd.DataFrame): データ。
        Mid (int, optional): M cell id. Defaults to 7.
        TCid (int, optional): TC cell id. Defaults to 5.

    Returns:
        pd.DataFrame: 選択したデータ。
    """
    Mid = 7
    TCid = 5
    dfq = g_dfraw.query("M_id=={} and TC_id=={}".format(Mid, TCid))
    return dfq


g_df_cell = choose_df(g_dfraw)
show_df_cell(g_dfraw, g_df_cell)
# 図の赤色部分


共通する特徴を求めます。

#### 頻出マイニングのmoduleのimport

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [ ]:
g_transaction = convert_to_transaction(g_df_cell, g_element_labels,)

In [ ]:
def get_freq_items(transaction, min_support=0.4):
    """頻出マイニングを行う

    Args:
        transaction ([[str]]): transaction.
        min_support (float, optional): minimum value of support ratio. Defaults to 0.4.

    Returns:
        pd.DataFrame: データ。
    """
    # min_threshold = 0.8
    te = TransactionEncoder()
    te.fit(transaction)
    te_ary = te.fit(transaction).transform(transaction)
    df = pd.DataFrame(te_ary, columns=te.columns_)
    df_freq_items = apriori(df, min_support=min_support,
                            max_len=10000, use_colnames=True, verbose=1)
    df_freq_items.sort_values(by="support", ascending=False, inplace=True)
    return df_freq_items


g_df_freq_items = get_freq_items(g_transaction)
g_df_freq_items.head(30)


組み合わせと頻度が出てくる。


- queryで用いたM_id, TC_idが現れる。
- Mn,Co,Fe元素が現れる。
- row_std_id==6が現れる。ー＞頻度のせいか？
- Rなどが現れない。

コメント：
たとえば、support=1の(TC_id==5, M_id==7)と(TC_id==5),(M_id==7)が分離して煩雑に出てきますが、まとめて、(TC_id==5, M_id==7)**のみ**出力する手法もあります。煩雑な出力はmlxtendにその手法が含まれていないせいです。

http://research.nii.ac.jp/~uno/code/lcm.html
はまとめて出力する手法を含んでいます。